# 01. Define and build genomic contexts

Build site- and gene-level annotations of the sorghum genome for downstream enrichment, variance-partitioning and phenotype-prediction analyses.


In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import socket
import sys
import json
from datetime import datetime
from time import perf_counter
import pandas as pd
import yaml

%load_ext autoreload
%autoreload 2

with open('config/project.yaml') as handle:
    project_config = yaml.safe_load(handle)
project_dir = project_config['remote']['project_root']
if os.path.realpath(os.getcwd()) != os.path.realpath(project_dir):
    raise ValueError(f'Working directory {os.getcwd()} must match configured project root {project_dir}.')
data_dir = f'{project_dir}/data'
source_dir = f'{data_dir}/source'
results_dir = f'{data_dir}/results'
os.makedirs(results_dir, exist_ok=True)
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

from src.plot import set_plot_style
from src.contexts import save_context_table
set_plot_style()

print(f'Host: {socket.gethostname()}')
print(f'Working directory: {os.getcwd()}')
print(f'Python: {sys.executable}')
print(f"Environment prefix: {os.environ.get('CONDA_PREFIX', '<not set>')}")

Host: rock01.local
Working directory: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum
Python: /lustre/home/dobby/Program/agent_program/micromamba-root/envs/dobby-base/bin/python
Environment prefix: /lustre/home/dobby/Program/agent_program/micromamba-root/envs/dobby-base


## 1. Define site-level contexts

Classify variants using genome structural annotations, epigenomic data and population-genetic metrics.



### 1.1 Variant preparation and filtering

The original sorghum population variant data have undergone quality control and genotype imputation. They include SNPs and indels, which are combined for subsequent analyses.

For the analyses here, retain genome-wide variants with MAF ≥0.05 and heterozygosity ≤0.05. Retain CDS variants with MAF ≥0.01 and the same heterozygosity limit.



In [2]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Preparing or reusing VCF inputs')

import pandas as pd
from src.vcf import merge_snp_indel_vcf, extract_samples_vcf

snp_vcf_file = f'{source_dir}/Sorghum_d8.noduplicates.allChr.snp._markernamesadded_imputed_snpeff.vcf.gz'
indel_vcf_file = f'{source_dir}/Sorghum_d8.noduplicates.allChr.indel._markernamesadded_imputed_snpeff.vcf.gz'
landrace_metadata_csv = f'{source_dir}/landraces_metadata.csv'
landrace_metadata_df = pd.read_csv(landrace_metadata_csv)

all_samples_vcf_file = f'{results_dir}/Sorghum_d8.allChr.imputed_snpeff.vcf.gz'
if not os.path.exists(all_samples_vcf_file):
    merge_snp_indel_vcf(snp_vcf_file, indel_vcf_file, all_samples_vcf_file)
else:
    print(f'Reusing merged VCF: {all_samples_vcf_file}')

landraces_vcf_file = f'{results_dir}/landraces.vcf.gz'
if not os.path.exists(landraces_vcf_file):
    sample_list = landrace_metadata_df['LIB'].tolist()
    if not pd.Index(sample_list).is_unique:
        raise ValueError('LIB sample IDs must be unique for genotype alignment.')
    extract_samples_vcf(all_samples_vcf_file, sample_list, landraces_vcf_file)
else:
    print(f'Reusing landrace VCF: {landraces_vcf_file}')

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[02:40:19] Preparing or reusing VCF inputs
Reusing merged VCF: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/Sorghum_d8.allChr.imputed_snpeff.vcf.gz
Reusing landrace VCF: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/landraces.vcf.gz
[02:40:19] Finished in 0.0s


In [3]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Preparing or reusing VCF inputs')


min_maf = 0.05
max_het = 0.05
n_jobs = 8

landraces_qc_vcf_file = f'{results_dir}/landraces_qc_maf{min_maf}_het{max_het}.vcf.gz'
landraces_qc_stat_h5 = f'{landraces_qc_vcf_file}.var_stat.h5'
if not os.path.exists(landraces_qc_vcf_file):
    from src.vcf import quality_control_vcf
    quality_control_vcf(
        landraces_vcf_file, landraces_qc_vcf_file, min_maf=min_maf, max_het=max_het,
        genome_fasta_file=f'{source_dir}/Sbicolor_730_v5.0.fa.gz', n_jobs=n_jobs,
    )
else:
    print(f'Reusing genome QC VCF: {landraces_qc_vcf_file}')
if not os.path.exists(landraces_qc_stat_h5):
    from src.vcf import build_var_stat_table
    build_var_stat_table(landraces_qc_vcf_file, landraces_qc_stat_h5, n_jobs=n_jobs)
else:
    print(f'Reusing variant statistics: {landraces_qc_stat_h5}')

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[02:40:20] Preparing or reusing VCF inputs
Reusing genome QC VCF: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/landraces_qc_maf0.05_het0.05.vcf.gz
Reusing variant statistics: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/landraces_qc_maf0.05_het0.05.vcf.gz.var_stat.h5
[02:40:20] Finished in 0.0s


The lower MAF cutoff in CDS regions retains more rare coding variants for coding-consequence and putative loss-of-function analyses.


In [4]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Preparing or reusing VCF inputs')

min_maf = 0.01
max_het = 0.05
n_jobs = 8

gff_file = f'{source_dir}/Sbicolor_730_v5.1.gene_exons.gff3.gz'
landraces_cds_vcf_file = f'{results_dir}/landraces_cds_maf{min_maf}_het{max_het}.vcf.gz'
landraces_cds_stat_h5 = f'{landraces_cds_vcf_file}.var_stat.h5'
if not os.path.exists(landraces_cds_vcf_file):
    from src.vcf import extract_cds_vcf
    extract_cds_vcf(
        landraces_vcf_file, gff_file, landraces_cds_vcf_file,
        min_maf=min_maf, max_het=max_het,
    )
else:
    print(f'Reusing CDS QC VCF: {landraces_cds_vcf_file}')
if not os.path.exists(landraces_cds_stat_h5):
    from src.vcf import build_var_stat_table
    build_var_stat_table(landraces_cds_vcf_file, landraces_cds_stat_h5, n_jobs=n_jobs)
else:
    print(f'Reusing variant statistics: {landraces_cds_stat_h5}')

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[02:40:20] Preparing or reusing VCF inputs
Reusing CDS QC VCF: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/landraces_cds_maf0.01_het0.05.vcf.gz
Reusing variant statistics: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/landraces_cds_maf0.01_het0.05.vcf.gz.var_stat.h5
[02:40:20] Finished in 0.0s


### 1.2 Annotate genome-wide sites with MAF ≥ 0.05

Annotate genome-wide variants with MAF ≥0.05. A variant can belong to multiple contexts.



In [5]:
from src.contexts import load_site_info_df
import pandas as pd

site_contexts_h5 = f'{results_dir}/site_contexts.genome.maf0.05.h5'

if not os.path.exists(site_contexts_h5):
    site_contexts_df = load_site_info_df(landraces_qc_stat_h5)
    site_contexts_df = site_contexts_df[['ID', 'CHROM', 'POS', 'MAF']]
    save_context_table(site_contexts_df, site_contexts_h5)
else:
    site_contexts_df = pd.read_hdf(site_contexts_h5, key='df')

if not site_contexts_df['ID'].is_unique:
    raise ValueError('Duplicate IDs would make context matching ambiguous.')
print(f'Table: {site_contexts_h5}; shape={site_contexts_df.shape}')

contexts_list = [i for i in site_contexts_df.columns.tolist() if i not in ['ID', 'CHROM', 'POS', 'MAF']]
print(f" {len(contexts_list)} contexts have already been recorded in site_contexts_df")

site_contexts_df

Table: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/site_contexts.genome.maf0.05.h5; shape=(6452577, 43)
 39 contexts have already been recorded in site_contexts_df


,ID,CHROM,POS,MAF,gene_body,genic_2kb,genic_proximal,genic_15kb,TE,repeat,...,low_recombination,med_recombination,high_recombination,maf_0.05_0.1,maf_0.1_0.2,maf_0.2_0.3,maf_0.3_0.4,maf_0.4_0.5,flowering_time_genes,core_flowering_time_genes
0,SNP_Chr01_20650,Chr01,20650,0.230248,False,True,True,True,False,False,...,False,False,False,False,False,True,False,False,False,False
1,SNP_Chr01_20658,Chr01,20658,0.229120,False,True,True,True,False,False,...,False,False,False,False,False,True,False,False,False,False
2,INDEL_Chr01_20702,Chr01,20702,0.230248,False,True,True,True,False,False,...,False,False,False,False,False,True,False,False,False,False
3,SNP_Chr01_20730,Chr01,20730,0.284424,False,True,True,True,False,False,...,False,False,False,False,False,True,False,False,False,False
4,INDEL_Chr01_20736,Chr01,20736,0.178330,False,True,True,True,False,False,...,False,False,False,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6452572,SNP_Chr10_62843420,Chr10,62843420,0.059819,True,True,True,True,False,False,...,False,False,False,True,False,False,False,False,False,False
6452573,SNP_Chr10_62843678,Chr10,62843678,0.075621,False,True,True,True,False,False,...,False,False,False,True,False,False,False,False,False,False
6452574,SNP_Chr10_62843803,Chr10,62843803,0.059819,False,True,True,True,False,False,...,False,False,False,True,False,False,False,False,False,False
6452575,SNP_Chr10_62843840,Chr10,62843840,0.058691,False,True,True,True,False,False,...,False,False,False,True,False,False,False,False,False,False


#### Genomic composition and gene structure

Distinguish gene bodies, gene neighborhoods extending 2, 5 or 15 kb on both sides, transposable elements (TEs) and other intergenic regions. TE annotations include DNA, LINE, LTR and SINE elements; other intergenic regions lie outside both genes with 2 kb flanks and TEs. These categories can overlap.

Within genes, distinguish untranslated regions (UTRs), coding sequences and introns using the longest transcript. Upstream and downstream regions extend 2 kb and follow gene orientation.


In [6]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

# genomic composition contexts

genomic_composition_contexts = ['gene_body', 'genic_2kb', 'genic_proximal', "genic_15kb", "TE", "other_intergenic"]
gff_file = f'{source_dir}/Sbicolor_730_v5.1.gene_exons.gff3.gz'
repeat_gff_file = f'{source_dir}/Sbicolor_730_v5.1.repeatmasked_assembly_v5.0.gff3.gz'

if not all([c in site_contexts_df.columns for c in genomic_composition_contexts]):
    from src.contexts import get_genomic_composition_contexts
    site_contexts_df = get_genomic_composition_contexts(site_contexts_df, gff_file, repeat_gff_file)
    save_context_table(site_contexts_df, site_contexts_h5)

# gene structure contexts

gene_structure_contexts = ['upstream_2kb', "five_prime_UTR", 'CDS', 'intron', "three_prime_UTR", 'downstream_2kb']
gff_file = f'{source_dir}/Sbicolor_730_v5.1.gene_exons.gff3.gz'

if not all([c in site_contexts_df.columns for c in gene_structure_contexts]):
    from src.contexts import get_gene_struc_contexts
    site_contexts_df = get_gene_struc_contexts(site_contexts_df, gff_file, gene_flank_size=2000)
    save_context_table(site_contexts_df, site_contexts_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[02:40:27] Annotating or reusing saved context columns
[02:40:27] Finished in 0.0s


#### Coding consequences

Use SnpEff predictions to distinguish putative loss-of-function, missense and synonymous variants. When several consequences are predicted for a variant, prioritize HIGH-impact consequences, followed by MODERATE-impact missense and LOW-impact synonymous consequences. These categories describe predicted effects.


In [7]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

coding_consequence_contexts = ['missense', 'synonymous', 'putative_LoF']
cds_snpeff_vcf_file = landraces_cds_vcf_file
if not all([c in site_contexts_df.columns for c in coding_consequence_contexts]):
    from src.contexts import get_coding_mutation_contexts
    site_contexts_df = get_coding_mutation_contexts(site_contexts_df, cds_snpeff_vcf_file)
    save_context_table(site_contexts_df, site_contexts_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[02:40:28] Annotating or reusing saved context columns
[02:40:28] Finished in 0.0s


#### Accessible chromatin regions (ACRs)

Identify variants within accessible chromatin regions using Supplementary Table 12 of Lu et al. (2019; DOI: 10.1038/s41477-019-0548-z). Coordinates have been converted to the sorghum V5 assembly.



In [8]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

acrs_contexts = ['ACR']
acr_bed_file = f'{source_dir}/ACRs.v5.bed.gz'
if not all([c in site_contexts_df.columns for c in acrs_contexts]):
    from src.contexts import get_bed_contexts
    site_contexts_df = get_bed_contexts(site_contexts_df, acr_bed_file, context_name='ACR')
    save_context_table(site_contexts_df, site_contexts_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[02:40:28] Annotating or reusing saved context columns
[02:40:28] Finished in 0.0s


#### Unmethylated regions (UMRs)

Identify variants within unmethylated regions using Dataset_S08 of Crisp et al. (2020; DOI: 10.1073/pnas.2010250117). Coordinates have been converted to the sorghum V5 assembly.



In [9]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

umrs_contexts = ['UMR']
umr_bed_file = f'{source_dir}/UMRs.v5.bed.gz'
if not all([c in site_contexts_df.columns for c in umrs_contexts]):
    from src.contexts import get_bed_contexts
    site_contexts_df = get_bed_contexts(site_contexts_df, umr_bed_file, context_name='UMR')
    save_context_table(site_contexts_df, site_contexts_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[02:40:29] Annotating or reusing saved context columns
[02:40:29] Finished in 0.0s


#### riboSNitch

Identify variants predicted to alter RNA secondary structure from SNPfold results. Classify variants with CC_BPPROB <0.8 as riboSNitches; when several predictions are available, use the lowest score. Variants with incomplete prediction information are excluded.


In [10]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

ribosnitch_contexts = ['riboSNitch']
snpfold_results_file = f'{source_dir}/snpfold.results.csv.gz'
if not all([c in site_contexts_df.columns for c in ribosnitch_contexts]):
    from src.contexts import get_ID_list_contexts
    snpfold_results_df = pd.read_csv(snpfold_results_file, sep=",", header=0)
    snpfold_results_df = snpfold_results_df[['variant_id', 'chrom', 'pos', 'CC_BPPROB']]
    snpfold_results_df.dropna(inplace=True, ignore_index=True)
    snpfold_results_df.rename(columns={'chrom': 'CHROM', 'pos': 'POS', 'variant_id': 'ID'}, inplace=True)
    # remove duplicates based on 'ID', keeping the smallest 'CC_BPPROB' value for each duplicate
    snpfold_results_df = snpfold_results_df.sort_values(by=['CC_BPPROB'], ascending=True).drop_duplicates(subset=['ID'], keep='first')
    snpfold_results_df.sort_values(by=['CHROM', 'POS'], inplace=True)
    snpfold_results_df['riboSNitch'] = snpfold_results_df.apply(lambda x: "riboSNitch" if x['CC_BPPROB'] < 0.8 else "non-riboSNitch", axis=1)
    ribosnitch_ids = snpfold_results_df[snpfold_results_df['riboSNitch'] == 'riboSNitch']['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, ribosnitch_ids, context_name='riboSNitch')
    save_context_table(site_contexts_df, site_contexts_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[02:40:29] Annotating or reusing saved context columns
[02:40:29] Finished in 0.0s


#### Phylogenetic conservation

Within genes and their 15 kb flanks on each side, classify sites as conserved (phyloP >1), neutral (−1 ≤ phyloP ≤1) or accelerated (phyloP <−1). Sites in these same gene neighborhoods without an evaluable phyloP score form the non_CEs category.



In [11]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

phylop_csv = f'{source_dir}/Sbi_PhyloP.csv.gz'
phylo_conservation_contexts = ['non_CEs', 'conserved_sites', 'neutral_sites', 'accelerated_sites']
if not all([c in site_contexts_df.columns for c in phylo_conservation_contexts]):
    from src.contexts import get_ID_list_contexts
    phylop_df = pd.read_csv(phylop_csv, sep=",", header=0)
    genic_site_ids = site_contexts_df[site_contexts_df['genic_15kb'] == True]['ID'].tolist()
    # conserved_sites
    conserved_site_ids = phylop_df[phylop_df['phyloP'] > 1]['ID'].tolist()
    conserved_site_ids = set(conserved_site_ids).intersection(set(genic_site_ids))
    site_contexts_df = get_ID_list_contexts(site_contexts_df, conserved_site_ids, context_name='conserved_sites')
    # neutral_sites
    neutral_site_ids = phylop_df[(phylop_df['phyloP'] >= -1) & (phylop_df['phyloP'] <= 1)]['ID'].tolist()
    neutral_site_ids = set(neutral_site_ids).intersection(set(genic_site_ids))
    site_contexts_df = get_ID_list_contexts(site_contexts_df, neutral_site_ids, context_name='neutral_sites')
    # accelerated_sites
    accelerated_site_ids = phylop_df[phylop_df['phyloP'] < -1]['ID'].tolist()
    accelerated_site_ids = set(accelerated_site_ids).intersection(set(genic_site_ids))
    site_contexts_df = get_ID_list_contexts(site_contexts_df, accelerated_site_ids, context_name='accelerated_sites')
    # other sites
    non_evaluable_site_ids = set(genic_site_ids) - set(conserved_site_ids) - set(neutral_site_ids) - set(accelerated_site_ids)
    site_contexts_df = get_ID_list_contexts(site_contexts_df, non_evaluable_site_ids, context_name='non_CEs')

    save_context_table(site_contexts_df, site_contexts_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[02:40:29] Annotating or reusing saved context columns
[02:40:29] Finished in 0.0s


#### Nucleotide diversity in gene neighborhoods

Calculate nucleotide diversity (π) in 500 bp windows within genes and their 15 kb flanks on each side. Retain windows containing at least three variants. Calculate quartiles across the retained gene-neighborhood windows and classify diversity as low (below Q25), intermediate (Q25 to below Q75) or high (Q75 and above).



In [12]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

gff_file = f'{source_dir}/Sbicolor_730_v5.1.gene_exons.gff3.gz'
genome_fasta_file = f'{source_dir}/Sbicolor_730_v5.0.fa.gz'
vcf_file = f'{results_dir}/landraces.vcf.gz'
pi_contexts = ['low_pi', 'med_pi', 'high_pi']
window_size = 500
gene_flank_size = 15000
min_variant_count = 3

if not all([c in site_contexts_df.columns for c in pi_contexts]):
    from src.contexts import get_pi_statistics, get_window_contexts
    pi_statistics_h5 = f'{results_dir}/pi_statistics.genic_15kb.b{window_size}.min_snp{min_variant_count}.h5'
    if not os.path.exists(pi_statistics_h5):
        pi_df = get_pi_statistics(
            gff_file, genome_fasta_file, vcf_file, window_size=window_size,
            gene_flank_size=gene_flank_size, min_snp_count=min_variant_count,
        )
        save_context_table(pi_df, pi_statistics_h5)
    else:
        pi_df = pd.read_hdf(pi_statistics_h5, key='df')
        print(f'Reusing window statistics: {pi_statistics_h5}; rows={len(pi_df):,}')
    low_threshold = pi_df['PI'].quantile(0.25)
    high_threshold = pi_df['PI'].quantile(0.75)
    low_pi_df = pi_df[pi_df['PI'] < low_threshold]
    med_pi_df = pi_df[(pi_df['PI'] >= low_threshold) & (pi_df['PI'] < high_threshold)]
    high_pi_df = pi_df[pi_df['PI'] >= high_threshold]
    site_contexts_df = get_window_contexts(site_contexts_df, low_pi_df, 'low_pi')
    site_contexts_df = get_window_contexts(site_contexts_df, med_pi_df, 'med_pi')
    site_contexts_df = get_window_contexts(site_contexts_df, high_pi_df, 'high_pi')
    save_context_table(site_contexts_df, site_contexts_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')


[02:40:30] Annotating or reusing saved context columns
[02:40:30] Finished in 0.0s


#### Genome-wide nucleotide diversity

Calculate π across the genome in 500 bp windows containing at least three variants. Define low, intermediate and high diversity using Q25 and Q75 calculated from these genome-wide windows.



In [13]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

gff_file = f'{source_dir}/Sbicolor_730_v5.1.gene_exons.gff3.gz'
genome_fasta_file = f'{source_dir}/Sbicolor_730_v5.0.fa.gz'
vcf_file = f'{results_dir}/landraces.vcf.gz'
recombination_file = f'{source_dir}/avg_cM_genomewide_NAM_population.csv.gz'
pi_contexts = ['low_pi_genome', 'med_pi_genome', 'high_pi_genome']
window_size = 500
min_variant_count = 3

if not all([c in site_contexts_df.columns for c in pi_contexts]):
    from src.contexts import get_genome_allele_statistics, get_window_contexts
    pi_statistics_h5 = f'{results_dir}/pi_statistics.genome.b{window_size}.min_snp{min_variant_count}.h5'
    if not os.path.exists(pi_statistics_h5):
        pi_df = get_genome_allele_statistics(
            genome_fasta_file, vcf_file, recombination_file, window_size=window_size,
            min_snp_count=min_variant_count, chunk_size=1_000_000, n_jobs=1,
        )
        save_context_table(pi_df, pi_statistics_h5)
    else:
        pi_df = pd.read_hdf(pi_statistics_h5, key='df')
        print(f'Reusing window statistics: {pi_statistics_h5}; rows={len(pi_df):,}')
    low_threshold = pi_df['PI'].quantile(0.25)
    high_threshold = pi_df['PI'].quantile(0.75)
    low_pi_df = pi_df[pi_df['PI'] < low_threshold]
    med_pi_df = pi_df[(pi_df['PI'] >= low_threshold) & (pi_df['PI'] < high_threshold)]
    high_pi_df = pi_df[pi_df['PI'] >= high_threshold]
    site_contexts_df = get_window_contexts(site_contexts_df, low_pi_df, 'low_pi_genome')
    site_contexts_df = get_window_contexts(site_contexts_df, med_pi_df, 'med_pi_genome')
    site_contexts_df = get_window_contexts(site_contexts_df, high_pi_df, 'high_pi_genome')
    save_context_table(site_contexts_df, site_contexts_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')


[02:40:30] Annotating or reusing saved context columns
[02:40:30] Finished in 0.0s


#### Recombination rate

Assign variants to 1 kb intervals of the recombination map and express rates in cM/Mb. Define low rates as <3 cM/Mb, intermediate rates as 3 to below Q75, and high rates as Q75 and above. Calculate Q75 among map intervals with rates >3 cM/Mb.


In [14]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

recombination_contexts = ['low_recombination', 'med_recombination', 'high_recombination']
recombination_file = f'{source_dir}/avg_cM_genomewide_NAM_population.csv.gz'

min_cM_per_Mb = 3

if not all([c in site_contexts_df.columns for c in recombination_contexts]):
    from src.contexts import get_site_recombination_contexts
    site_contexts_df = get_site_recombination_contexts(site_contexts_df, recombination_file, min_CmPerMb=min_cM_per_Mb)
    save_context_table(site_contexts_df, site_contexts_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[02:40:30] Annotating or reusing saved context columns
[02:40:30] Finished in 0.0s


#### Minor allele frequency

Group variants by minor allele frequency (MAF) to distinguish frequency classes. Each interval includes its lower boundary and excludes its upper boundary, except that the final genome-wide interval includes MAF =0.5.


In [15]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

maf_contexts = ['maf_0.05_0.1', 'maf_0.1_0.2', 'maf_0.2_0.3', 'maf_0.3_0.4', 'maf_0.4_0.5']

if not all([c in site_contexts_df.columns for c in maf_contexts]):
    from src.contexts import get_ID_list_contexts
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.05) & (site_contexts_df['MAF'] < 0.1)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.05_0.1')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.1) & (site_contexts_df['MAF'] < 0.2)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.1_0.2')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.2) & (site_contexts_df['MAF'] < 0.3)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.2_0.3')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.3) & (site_contexts_df['MAF'] < 0.4)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.3_0.4')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.4) & (site_contexts_df['MAF'] <= 0.5)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.4_0.5')
    save_context_table(site_contexts_df, site_contexts_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[02:40:31] Annotating or reusing saved context columns
[02:40:31] Finished in 0.0s


#### Flowering-time genes

Identify variants within catalogued flowering-time genes and their 2 kb flanks. The core subset consists of genes curated from the literature.


In [16]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

flowering_genes_file = f'{source_dir}/Sorghum_flowering_genes.csv'
flowering_genes_contexts = ['flowering_time_genes', 'core_flowering_time_genes']

if not all([c in site_contexts_df.columns for c in flowering_genes_contexts]):
    from src.contexts import get_gene_contexts
    flowering_genes_df = pd.read_csv(flowering_genes_file, sep=",", header=0)
    flowering_gene_ids = flowering_genes_df['Gene_ID'].to_list()
    curated_flowering_gene_ids = flowering_genes_df.loc[flowering_genes_df['Annotation_source'] == 'Literature-curated genes', 'Gene_ID'].to_list()
    site_contexts_df = get_gene_contexts(site_contexts_df, flowering_gene_ids, gff_file, 'flowering_time_genes', flank_size=2000)
    site_contexts_df = get_gene_contexts(site_contexts_df, curated_flowering_gene_ids, gff_file, 'core_flowering_time_genes', flank_size=2000)
    save_context_table(site_contexts_df, site_contexts_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[02:40:31] Annotating or reusing saved context columns
[02:40:31] Finished in 0.0s


### 1.3 Annotate CDS sites with MAF ≥ 0.01

Use CDS variants with MAF ≥0.01 to describe coding variation, including rarer variants than the genome-wide variants.



In [17]:
from src.contexts import load_site_info_df
import pandas as pd

site_contexts_h5 = f'{results_dir}/site_contexts.cds.maf0.01.h5'

if not os.path.exists(site_contexts_h5):
    site_contexts_df = load_site_info_df(landraces_cds_stat_h5)
    site_contexts_df = site_contexts_df[['ID', 'CHROM', 'POS', 'MAF']]
    save_context_table(site_contexts_df, site_contexts_h5)
else:
    site_contexts_df = pd.read_hdf(site_contexts_h5, key='df')

if not site_contexts_df['ID'].is_unique:
    raise ValueError('Duplicate IDs would make context matching ambiguous.')
print(f'Table: {site_contexts_h5}; shape={site_contexts_df.shape}')

contexts_list = [i for i in site_contexts_df.columns.tolist() if i not in ['ID', 'CHROM', 'POS', 'MAF']]
print(f" {len(contexts_list)} contexts have already been recorded in site_contexts_df")

site_contexts_df

Table: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/site_contexts.cds.maf0.01.h5; shape=(386925, 16)
 12 contexts have already been recorded in site_contexts_df


,ID,CHROM,POS,MAF,missense_cds_0.01,synonymous_cds_0.01,putative_LoF_cds_0.01,maf_0.01_0.03_cds_0.01,maf_0.03_0.05_cds_0.01,maf_0.05_0.1_cds_0.01,maf_0.1_0.2_cds_0.01,maf_0.2_0.3_cds_0.01,maf_0.3_0.4_cds_0.01,maf_0.4_0.5_cds_0.01,flowering_time_genes,core_flowering_time_genes
0,SNP_Chr01_22482,Chr01,22482,0.019187,True,False,False,True,False,False,False,False,False,False,False,False
1,SNP_Chr01_22585,Chr01,22585,0.019187,False,True,False,True,False,False,False,False,False,False,False,False
2,SNP_Chr01_22703,Chr01,22703,0.019187,True,False,False,True,False,False,False,False,False,False,False,False
3,SNP_Chr01_33180,Chr01,33180,0.024831,False,True,False,True,False,False,False,False,False,False,False,False
4,SNP_Chr01_34142,Chr01,34142,0.382619,True,False,False,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
386920,SNP_Chr10_62837224,Chr10,62837224,0.051919,False,True,False,False,False,True,False,False,False,False,False,False
386921,SNP_Chr10_62841974,Chr10,62841974,0.453725,True,False,False,False,False,False,False,False,False,True,False,False
386922,SNP_Chr10_62842354,Chr10,62842354,0.016930,False,True,False,True,False,False,False,False,False,False,False,False
386923,INDEL_Chr10_62842730,Chr10,62842730,0.030474,False,False,True,False,True,False,False,False,False,False,False,False


#### Coding consequences

Classify CDS variants as putative loss-of-function, missense or synonymous using the same SnpEff consequence priorities as above.


In [18]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

coding_consequence_contexts = ['missense_cds_0.01', 'synonymous_cds_0.01', 'putative_LoF_cds_0.01']
cds_snpeff_vcf_file = landraces_cds_vcf_file
if not all([c in site_contexts_df.columns for c in coding_consequence_contexts]):
    from src.contexts import get_coding_mutation_contexts
    site_contexts_df = get_coding_mutation_contexts(site_contexts_df, cds_snpeff_vcf_file, dataset='_cds_0.01')
    save_context_table(site_contexts_df, site_contexts_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[02:40:32] Annotating or reusing saved context columns
[02:40:32] Finished in 0.0s


#### Minor allele frequency

Divide CDS variants into seven MAF classes. Each interval includes its lower boundary and excludes its upper boundary, except that the final interval includes MAF =0.5 (0.4 ≤ MAF ≤0.5).


In [19]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

maf_contexts = ['maf_0.01_0.03_cds_0.01', 'maf_0.03_0.05_cds_0.01', 'maf_0.05_0.1_cds_0.01', 'maf_0.1_0.2_cds_0.01', 'maf_0.2_0.3_cds_0.01', 'maf_0.3_0.4_cds_0.01', 'maf_0.4_0.5_cds_0.01']

if not all([c in site_contexts_df.columns for c in maf_contexts]):
    from src.contexts import get_ID_list_contexts
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.01) & (site_contexts_df['MAF'] < 0.03)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.01_0.03_cds_0.01')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.03) & (site_contexts_df['MAF'] < 0.05)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.03_0.05_cds_0.01')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.05) & (site_contexts_df['MAF'] < 0.1)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.05_0.1_cds_0.01')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.1) & (site_contexts_df['MAF'] < 0.2)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.1_0.2_cds_0.01')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.2) & (site_contexts_df['MAF'] < 0.3)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.2_0.3_cds_0.01')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.3) & (site_contexts_df['MAF'] < 0.4)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.3_0.4_cds_0.01')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.4) & (site_contexts_df['MAF'] <= 0.5)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.4_0.5_cds_0.01')
    save_context_table(site_contexts_df, site_contexts_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[02:40:32] Annotating or reusing saved context columns
[02:40:32] Finished in 0.0s


#### Flowering-time genes

Identify CDS variants in the same flowering-time gene catalog and its literature-curated core subset, using the same 2 kb flanks as above.


In [20]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

flowering_genes_file = f'{source_dir}/Sorghum_flowering_genes.csv'
flowering_genes_contexts = ['flowering_time_genes', 'core_flowering_time_genes']

if not all([c in site_contexts_df.columns for c in flowering_genes_contexts]):
    from src.contexts import get_gene_contexts
    flowering_genes_df = pd.read_csv(flowering_genes_file, sep=",", header=0)
    flowering_gene_ids = flowering_genes_df['Gene_ID'].to_list()
    curated_flowering_gene_ids = flowering_genes_df.loc[flowering_genes_df['Annotation_source'] == 'Literature-curated genes', 'Gene_ID'].to_list()
    site_contexts_df = get_gene_contexts(site_contexts_df, flowering_gene_ids, gff_file, 'flowering_time_genes', flank_size=2000)
    site_contexts_df = get_gene_contexts(site_contexts_df, curated_flowering_gene_ids, gff_file, 'core_flowering_time_genes', flank_size=2000)
    save_context_table(site_contexts_df, site_contexts_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[02:40:33] Annotating or reusing saved context columns
[02:40:33] Finished in 0.0s


## 2. Define gene-level contexts

Classify genes by their roles in flowering time, expression profiles, co-expression networks, gene families and recombination environment.


Use annotated genes on the reference chromosomes as the basis for gene-level classification.


In [21]:
import pandas as pd

gff_file = f'{source_dir}/Sbicolor_730_v5.1.gene_exons.gff3.gz'
gene_contexts_h5 = f'{results_dir}/gene_contexts.h5'

if not os.path.exists(gene_contexts_h5):
    from src.contexts import load_gene_info_df
    gene_contexts_df = load_gene_info_df(gff_file)
    save_context_table(gene_contexts_df, gene_contexts_h5)
else:
    gene_contexts_df = pd.read_hdf(gene_contexts_h5, key='df')

if not gene_contexts_df['ID'].is_unique:
    raise ValueError('Duplicate IDs would make context matching ambiguous.')
print(f'Table: {gene_contexts_h5}; shape={gene_contexts_df.shape}')

contexts_list = [i for i in gene_contexts_df.columns.tolist() if i not in ['ID', 'CHROM', 'START', 'END', 'STRAND']]
print(f" {len(contexts_list)} contexts have already been recorded in gene_contexts_df")

gene_contexts_df

Table: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/gene_contexts.h5; shape=(31819, 18)
 13 contexts have already been recorded in gene_contexts_df


,ID,CHROM,START,END,STRAND,flowering_time_genes,no_expression_genes,high_variance_genes,stably_expressed_genes,module_hub_genes,module_genes,non_module_genes,orphan_genes,single_copy_genes,multi_copy_genes,low_recombination_genes,med_recombination_genes,high_recombination_genes
0,Sobic.001G000100.v5.1,Chr01,22085,23338,+,False,False,False,False,False,True,False,False,True,False,True,False,False
1,Sobic.001G000200.v5.1,Chr01,31590,35375,-,False,False,False,False,True,False,False,False,True,False,True,False,False
2,Sobic.001G000300.v5.1,Chr01,43890,44492,-,False,False,False,False,False,True,False,False,False,True,True,False,False
3,Sobic.001G000400.v5.1,Chr01,44828,63076,-,False,False,False,False,False,True,False,False,True,False,True,False,False
4,Sobic.001G000700.v5.1,Chr01,80708,83931,+,False,False,False,False,False,True,False,False,False,True,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31814,Sobic.010G279800.v5.1,Chr10,62816416,62825451,+,False,False,False,False,False,True,False,False,True,False,False,False,False
31815,Sobic.010G279900.v5.1,Chr10,62825908,62834521,-,False,False,False,False,False,True,False,False,True,False,False,False,False
31816,Sobic.010G280000.v5.1,Chr10,62836480,62837370,-,False,False,False,False,False,True,False,False,True,False,False,False,False
31817,Sobic.010G280100.v5.1,Chr10,62838030,62841326,+,False,False,False,False,False,True,False,False,True,False,False,False,False


#### Flowering-time genes

Identify genes included in the flowering-time gene catalog.


In [22]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

flowering_gene_contexts = ['flowering_time_genes']

if not all([c in gene_contexts_df.columns for c in flowering_gene_contexts]):
    from src.contexts import get_ID_list_contexts
    flowering_genes_file = f'{source_dir}/Sorghum_flowering_genes.csv'
    flowering_genes_df = pd.read_csv(flowering_genes_file, sep=",", header=0)
    flowering_gene_ids = flowering_genes_df['Gene_ID'].to_list()
    gene_contexts_df = get_ID_list_contexts(gene_contexts_df, flowering_gene_ids, context_name='flowering_time_genes')
    save_context_table(gene_contexts_df, gene_contexts_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[02:40:33] Annotating or reusing saved context columns
[02:40:33] Finished in 0.0s


#### Gene expression

Distinguish genes with no detected expression, high expression variability or stable expression across the analyzed samples. Treat TPM ≤2 as undetected. Account for the relationship between mean expression and variance using LOWESS; standardized residuals >1 identify high-variability genes. Stable genes have residuals <1 and detectable expression in every sample.



In [23]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

expression_gene_contexts = ['no_expression_genes', 'high_variance_genes', 'stably_expressed_genes']

min_expr = 2.0
high_var_threshold = 1.0

if not all([c in gene_contexts_df.columns for c in expression_gene_contexts]):
    from src.contexts import get_expression_contexts
    expression_tpm_csv = f'{source_dir}/Sbi_NCBI_SRA_TPMs.csv.gz'
    gene_contexts_df = get_expression_contexts(
        gene_contexts_df, expression_tpm_csv, min_expr=min_expr, high_var_threshold=high_var_threshold,
    )
    save_context_table(gene_contexts_df, gene_contexts_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[02:40:34] Annotating or reusing saved context columns
[02:40:34] Finished in 0.0s


#### Co-expression networks

Distinguish genes outside co-expression modules, non-hub module members and module hubs.



In [24]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

network_gene_contexts = ['non_module_genes', 'module_genes', 'module_hub_genes']

edge_cutoff = 0.1

if not all([c in gene_contexts_df.columns for c in network_gene_contexts]):
    from src.contexts import get_GCNs_contexts
    module_file = f'{source_dir}/edge_weight_N3.0.01.clusterone.txt.gz'
    edge_weight_file = f'{source_dir}/edge_weight_N3.0.01.tsv.gz'
    gene_contexts_df = get_GCNs_contexts(gene_contexts_df, edge_weight_file, module_file, cut_off=edge_cutoff)
    save_context_table(gene_contexts_df, gene_contexts_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[02:40:34] Annotating or reusing saved context columns
[02:40:34] Finished in 0.0s


#### Gene families

Use the Poaceae hierarchical orthologous groups to distinguish groups containing one or multiple sorghum genes. Classify their members as single-copy or multiple-copy genes, respectively. Genes without a group assignment form the orphan category.


In [25]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

gene_family_contexts = ['orphan_genes', 'single_copy_genes', 'multi_copy_genes']

if not all([c in gene_contexts_df.columns for c in gene_family_contexts]):
    from src.contexts import get_gene_family_contexts
    gene_family_file = f'{source_dir}/Poaceae_HOG.tsv.gz'
    gene_contexts_df = get_gene_family_contexts(gene_contexts_df, gene_family_file)
    save_context_table(gene_contexts_df, gene_contexts_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[02:40:34] Annotating or reusing saved context columns
[02:40:34] Finished in 0.0s


#### Gene recombination

Estimate each gene’s recombination rate as the mean rate of overlapping 1 kb map intervals, without weighting by overlap length. Among genes with mapped rates, define low, intermediate and high categories using Q25 and Q75.



In [26]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

recombination_contexts = ['low_recombination_genes', 'med_recombination_genes', 'high_recombination_genes']

if not all([c in gene_contexts_df.columns for c in recombination_contexts]):
    from src.contexts import get_gene_recombination_contexts
    recombination_file = f'{source_dir}/avg_cM_genomewide_NAM_population.csv.gz'
    gene_contexts_df = get_gene_recombination_contexts(gene_contexts_df, recombination_file)
    save_context_table(gene_contexts_df, gene_contexts_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[02:40:35] Annotating or reusing saved context columns
[02:40:35] Finished in 0.0s


## 3. Summarize genomic contexts

Compare the numbers and proportions of variants or genes assigned to each context in the genome-wide, CDS and gene datasets. The denominator is the total number of variants or genes in each dataset.



In [27]:
import json

datasets = {
    'genome': f'{results_dir}/site_contexts.genome.maf0.05.h5', # All variants with MAF >= 0.05
    'cds': f'{results_dir}/site_contexts.cds.maf0.01.h5', # All variants in coding regions with MAF >= 0.01
    'gene': f'{results_dir}/gene_contexts.h5' # All genes
}

contexts_settings_json = f'{project_dir}/config/contexts_settings.json'
with open(contexts_settings_json, "r") as f:
    contexts_settings = json.load(f)

### 3.1 Genome sites with MAF ≥ 0.05

In [28]:
dataset = 'genome'
context_df = pd.read_hdf(datasets[dataset], key='df')
context_ids = list(contexts_settings[dataset])
summary_df = context_df[context_ids].sum().rename('count').to_frame()
summary_df.index.name = 'context'
summary_df['count'] = summary_df['count'].astype(int)
summary_df['total'] = len(context_df)
summary_df['fraction'] = summary_df['count'] / len(context_df)
summary_file = f'{results_dir}/contexts.{dataset}.summary.csv.gz'
summary_df.reset_index().to_csv(summary_file, index=False, compression='gzip')
print(f'Summary: {summary_file}')
print(f'Rows: {len(context_df):,}')
summary_df

Summary: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/contexts.genome.summary.csv.gz
Rows: 6,452,577


,count,total,fraction
context,,,
gene_body,1013897,6452577,0.157131
genic_2kb,2277703,6452577,0.352991
genic_proximal,3077170,6452577,0.476890
genic_15kb,3938903,6452577,0.610439
TE,3736513,6452577,0.579073
other_intergenic,1054970,6452577,0.163496
upstream_2kb,888419,6452577,0.137684
five_prime_UTR,87769,6452577,0.013602
CDS,240922,6452577,0.037337


### 3.2 CDS sites with MAF ≥ 0.01

In [29]:
dataset = 'cds'
context_df = pd.read_hdf(datasets[dataset], key='df')
context_ids = list(contexts_settings[dataset])
summary_df = context_df[context_ids].sum().rename('count').to_frame()
summary_df.index.name = 'context'
summary_df['count'] = summary_df['count'].astype(int)
summary_df['total'] = len(context_df)
summary_df['fraction'] = summary_df['count'] / len(context_df)
summary_file = f'{results_dir}/contexts.{dataset}.summary.csv.gz'
summary_df.reset_index().to_csv(summary_file, index=False, compression='gzip')
print(f'Summary: {summary_file}')
print(f'Rows: {len(context_df):,}')
summary_df

Summary: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/contexts.cds.summary.csv.gz
Rows: 386,925


,count,total,fraction
context,,,
missense_cds_0.01,186477,386925,0.481946
synonymous_cds_0.01,174908,386925,0.452046
putative_LoF_cds_0.01,12820,386925,0.033133
maf_0.01_0.03_cds_0.01,107212,386925,0.277087
maf_0.03_0.05_cds_0.01,38791,386925,0.100255
maf_0.05_0.1_cds_0.01,56971,386925,0.147240
maf_0.1_0.2_cds_0.01,70340,386925,0.181792
maf_0.2_0.3_cds_0.01,44412,386925,0.114782
maf_0.3_0.4_cds_0.01,35514,386925,0.091785


### 3.3 Gene-level contexts

In [30]:
dataset = 'gene'
context_df = pd.read_hdf(datasets[dataset], key='df')
context_ids = list(contexts_settings[dataset])
summary_df = context_df[context_ids].sum().rename('count').to_frame()
summary_df.index.name = 'context'
summary_df['count'] = summary_df['count'].astype(int)
summary_df['total'] = len(context_df)
summary_df['fraction'] = summary_df['count'] / len(context_df)
summary_file = f'{results_dir}/contexts.{dataset}.summary.csv.gz'
summary_df.reset_index().to_csv(summary_file, index=False, compression='gzip')
print(f'Summary: {summary_file}')
print(f'Rows: {len(context_df):,}')
summary_df

Summary: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/contexts.gene.summary.csv.gz
Rows: 31,819


,count,total,fraction
context,,,
flowering_time_genes,312,31819,0.009805
no_expression_genes,1232,31819,0.038719
high_variance_genes,5195,31819,0.163267
stably_expressed_genes,1350,31819,0.042427
non_module_genes,4267,31819,0.134102
module_genes,20419,31819,0.641723
module_hub_genes,7133,31819,0.224174
orphan_genes,3237,31819,0.101732
multi_copy_genes,12421,31819,0.390364
